In [58]:
#import libraries
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

In [59]:
#import table
conn = sqlite3.connect(r"F:\VScode L.3\The final project\City-Library-After-School-Program\library.db")

# tables overview
books = pd.read_sql_query("SELECT * FROM books;",conn)
checkouts = pd.read_sql_query("SELECT * FROM checkouts;",conn)
members = pd.read_sql_query("SELECT * FROM members;",conn)

print("BOOKS: \n",books)
print("------------------------------------")
print("CHECKOUT: \n",checkouts)
print("------------------------------------")
print("MEMBERS: \n",members)

BOOKS: 
     book_id                    title          author
0       501          The Silver Kite   Amina Darwish
1       502           Desert Compass   Amina Darwish
2       503        The Lantern Maker    Adel Roushdy
3       504      Rooftop Astronomers    Adel Roushdy
4       505      Letters to the Nile       Aya Hafez
5       506      The Paper Boat Club       Aya Hafez
6       507    Fossils and Fireflies     Dalia Serry
7       508    The Quiet Observatory     Dalia Serry
8       509      Marbles and Mirrors     Diaa Sultan
9       510    The Missing Metronome     Diaa Sultan
10      511     Winter in Alexandria    Farida Anwar
11      512   The Cartographer's Cat    Farida Anwar
12      513   Circuits for Beginners    Galal Mounir
13      514    The Robot in Row Nine    Galal Mounir
14      515       Songs of the Oasis      Hoda Bakry
15      516  The Beekeeper's Almanac      Hoda Bakry
16      517  Shadows on the Corniche     Hani Nagati
17      518     The Ninth Lighthouse 

In [107]:
#Answer SQL questions (code-version)
#-------------------------------------
##1.How much is each member borrowing?
q1 = pd.read_sql_query("""SELECT members.member_id,
members.first_name,
members.last_name,
 COUNT(checkouts.checkout_id) AS total_checkouts 
 FROM  members
 LEFT JOIN  checkouts               
 ON checkouts.member_id = members.member_id 
 GROUP BY members.member_id,
members.first_name,
members.last_name
ORDER BY total_checkouts DESC;""", conn)
#LEFT JOIN >> to count all members, even those who have not checkouts

print("1. How much is each member borrowing? \n",q1)
print("------------------------------------")

#-------------------------------------
##2.Which books match a chosen author pattern?

q2 = pd.read_sql_query("""SELECT title,
book_id,
author
FROM books
WHERE author LIKE 's%'
ORDER BY title ;""", conn)

print("2. Which books match a chosen author pattern? \n",q2)
print("------------------------------------")

#-------------------------------------
##3.What are the most popular books?

q3 = pd.read_sql_query("""SELECT books.title,
books.book_id,
COUNT(checkouts.checkout_id) AS total_checkouts
FROM checkouts
JOIN books
ON checkouts.book_id = books.book_id
GROUP BY books.book_id, books.title
ORDER BY total_checkouts DESC
LIMIT 5;""", conn)

print("3. What are the most popular books? \n",q3)
print("------------------------------------")


#-------------------------------------
##4.Who are the most active readers?

q4 = pd.read_sql_query("""SELECT members.first_name,
members.last_name,
COUNT(checkouts.checkout_id) AS total_checkouts
FROM checkouts
LEFT JOIN members
ON checkouts.member_id = members.member_id
GROUP BY members.member_id, members.first_name, members.last_name
ORDER BY total_checkouts DESC
LIMIT 10;""", conn)

print("4. Who are the most active readers? \n",q4)
print("------------------------------------")

#-------------------------------------
##5.What does a neighborhood's activity look like further back in time?

q5 = pd.read_sql_query("""SELECT members.neighborhood,
checkouts.checkout_id,
checkouts.member_id,
checkouts.book_id,
checkouts.checkout_date,
checkouts.return_date
FROM checkouts
JOIN members
ON checkouts.member_id = members.member_id
WHERE LOWER(members.neighborhood) = 'shubra'
ORDER BY checkouts.checkout_date DESC
LIMIT -1 OFFSET 10;""", conn)

print("5. What does a neighborhood's activity look like further back in time? \n",q5)
print("------------------------------------")


1. How much is each member borrowing? 
     member_id first_name last_name  total_checkouts
0        1034        Aya     Wahba               25
1        1044     Sherif     Saleh               21
2        1008       Ziad     Saleh               19
3        1010       Nour     Nabil               18
4        1027    Mostafa     Fouad               18
..        ...        ...       ...              ...
75       1063      Layla     Fouad                0
76       1064      Fares     Sabry                0
77       1066       Amir     Wahba                0
78       1069     Bassel      Adel                0
79       1078     Habiba     Osman                0

[80 rows x 4 columns]
------------------------------------
2. Which books match a chosen author pattern? 
                     title  book_id        author
0  Riddles of the Red Sea      529  Sara Tantawy
1       The Last Bookmark      532   Samir Zohdy
2       The Sandstone Key      530  Sara Tantawy
3   Voices in the Library      5

In [61]:
# SQL to DF

library_df = pd.read_sql_query("""
SELECT
    checkouts.checkout_id,
    checkouts.member_id,
    checkouts.book_id,
    checkouts.checkout_date,
    checkouts.return_date,
    members.first_name,
    members.last_name,
    members.grade,
    members.neighborhood,
    members.membership_status,
    members.join_date,
    books.title,
    books.author
FROM checkouts
JOIN members
    ON checkouts.member_id = members.member_id
JOIN books
    ON checkouts.book_id = books.book_id;
""", conn)

library_df.index.name = None
library_df.columns.name = None


print("LIBRARY merged_DB:\n", library_df )



library_df.to_csv("library_df.csv",index=False)



LIBRARY merged_DB:
      checkout_id  member_id  book_id checkout_date return_date first_name  \
0           9263       1047      517    2024-10-21  2024-11-07       Sara   
1           9340       1072      513    2025-08-24  2025-09-01       Seif   
2           9231       1053      523    2024-02-04  2024-02-16       Adam   
3           9129       1032      513    2025-06-21  2025-06-29       Nada   
4           9370       1079      511    2025-11-11  2025-12-03       Rana   
..           ...        ...      ...           ...         ...        ...   
386         9232       1044      513    2025-05-26  2025-06-11     Sherif   
387         9084       1008      511    2024-06-27  2024-07-22       Ziad   
388         9116       1024      519    2025-01-10  2025-02-09    Youssef   
389         9352       1076      501    2025-02-02         NaN       Dina   
390         9246       1044      529    2025-06-25  2025-07-04     Sherif   

    last_name  grade neighborhood membership_status   j

In [62]:
book_catalog= pd.read_json("books.json")
library_df= pd.read_csv(r"f:\VScode L.3\The final project\City-Library-After-School-Program\library_df.csv")

print(library_df["book_id"].dtype)
print(book_catalog["book_id"].dtype)

print(book_catalog["book_id"].duplicated().sum())

merged_detailed_data = library_df.merge (book_catalog, on="book_id",validate="many_to_one")

print("BEFORE:", len(library_df))
print("AFTER:", len(merged_detailed_data))

merged_detailed_data.to_csv("Merged (sql-json).csv",index=False)

int64
int64
0
BEFORE: 391
AFTER: 391


In [63]:
with open(r"F:\VScode L.3\The final project\City-Library-After-School-Program\summer_checkouts.html","r",encoding="utf-8") as file:
    summer_checkouts = file.read()

soup = BeautifulSoup(summer_checkouts,"html.parser")
print(soup.find_all("table"))

summer_checkouts = pd.read_html(r"F:\VScode L.3\The final project\City-Library-After-School-Program\summer_checkouts.html")
summer_checkouts = summer_checkouts[0]

summer_checkouts.columns = (
    summer_checkouts.columns
    .str.lower()
    .str.replace(" ", "_")
)


summer_checkouts.to_csv("scraped_data HTML.csv", index=False)

summer_checkouts_df = pd.read_csv(r"F:\VScode L.3\The final project\City-Library-After-School-Program\scraped_data HTML.csv")



print(summer_checkouts_df)

[<table>
<tr><th>Member ID</th><th>Book ID</th><th>Checkout Date</th></tr>
<tr><td>1026</td><td>522</td><td>2025-07-11</td></tr>
<tr><td>1049</td><td>520</td><td>2025-07-11</td></tr>
<tr><td>1062</td><td>525</td><td>2025-07-05</td></tr>
<tr><td>1065</td><td>520</td><td>2025-07-07</td></tr>
<tr><td>1104</td><td>515</td><td>2025-07-07</td></tr>
<tr><td>1009</td><td>503</td><td>2025-07-09</td></tr>
<tr><td>1063</td><td>522</td><td>2025-07-07</td></tr>
<tr><td>1022</td><td>511</td><td>2025-07-12</td></tr>
<tr><td>1029</td><td>523</td><td>2025-07-09</td></tr>
<tr><td>1201</td><td>509</td><td>2025-07-10</td></tr>
<tr><td>1005</td><td>513</td><td>2025-07-10</td></tr>
<tr><td>1104</td><td>526</td><td>2025-07-05</td></tr>
<tr><td>1058</td><td>518</td><td>2025-07-08</td></tr>
<tr><td>1002</td><td>521</td><td>2025-07-08</td></tr>
<tr><td>1150</td><td>530</td><td>2025-07-10</td></tr>
<tr><td>1041</td><td>504</td><td>2025-07-05</td></tr>
<tr><td>1055</td><td>526</td><td>2025-07-11</td></tr>
<tr><td

In [64]:
summer_checkouts_df = pd.read_csv(r"F:\VScode L.3\The final project\City-Library-After-School-Program\scraped_data HTML.csv")
merged_detailed_data = pd.read_csv(r"F:\VScode L.3\The final project\City-Library-After-School-Program\Merged (sql-json).csv")

#Checking the column names to ensure matching merging keys
print(merged_detailed_data.columns.tolist())
print(summer_checkouts_df.columns.tolist())


final_df = merged_detailed_data.merge(
    summer_checkouts_df,
    on=["member_id", "book_id", "checkout_date"],
    how="left",
    validate="many_to_one"
)

final_df.to_csv("task1_combined_data.csv", index=False)

['checkout_id', 'member_id', 'book_id', 'checkout_date', 'return_date', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'join_date', 'title', 'author', 'genre', 'pages', 'publication_year', 'publisher']
['member_id', 'book_id', 'checkout_date']
